# 02. Data Preprocessing (PySpark)

Notebook ini memuat seluruh logika transformasi *Big Data*. Mulai dari membaca *GeoJSON* dari MongoDB, *flattening*, perbaikan anomali geometri bumi (Trigonometri 3D), hingga *Feature Engineering* yang siap dipakai oleh algoritma K-Means.

### Tahap 1: Import Library & Load Konfigurasi
Menyiapkan seluruh alat dari *PySpark ML* dan fungsi SQL, serta membaca pengaturan `.env`.

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, split, cos, sin, radians, log1p, to_timestamp
from pyspark.sql.types import DoubleType, TimestampType
from pyspark.ml.feature import VectorAssembler, StandardScaler

SPARK_MASTER = 'local[*]'
MONGO_URI = 'mongodb://localhost:27017'
MONGO_DB = 'earthquake_db'
MONGO_RAW_COL = 'raw_earthquakes_emsc'
MONGO_CLEAN_COL = 'clean_earthquakes_emsc'
FEATURE_COLS = ['x','y','z','depth_log','mag']


### Tahap 2: Menyalakan Spark Session & Konektor MongoDB
Membentuk sesi komputasi terdistribusi dan secara spesifik menyuntikkan *library* konektor MongoDB versi 10.3.0 agar kompatibel dengan PySpark 3.5.0.

In [22]:
spark = SparkSession.builder \
    .appName("EarthquakePreprocessing") \
    .master(SPARK_MASTER) \
    .config("spark.mongodb.read.connection.uri", MONGO_URI) \
    .config("spark.mongodb.write.connection.uri", MONGO_URI) \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Session Aktif di Master URL: {SPARK_MASTER}")

Spark Session Aktif di Master URL: local[*]


### Tahap 3: Membaca Data Mentah
Menyedot seluruh dokumen dari MongoDB (*Collection: raw_earthquakes*) dan memuatnya ke dalam memori DataFrame Spark.

In [23]:
df_raw = spark.read.format("mongodb") \
    .option("database", MONGO_DB) \
    .option("collection", MONGO_RAW_COL) \
    .load()

print(f"Total Data Mentah: {df_raw.count()} baris")
df_raw.printSchema()

Total Data Mentah: 141740 baris
root
 |-- _id: string (nullable = true)
 |-- geometry: struct (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- coordinates: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- id: string (nullable = true)
 |-- properties: struct (nullable = true)
 |    |-- source_id: string (nullable = true)
 |    |-- source_catalog: string (nullable = true)
 |    |-- lastupdate: string (nullable = true)
 |    |-- time: string (nullable = true)
 |    |-- flynn_region: string (nullable = true)
 |    |-- lat: double (nullable = true)
 |    |-- lon: double (nullable = true)
 |    |-- depth: double (nullable = true)
 |    |-- evtype: string (nullable = true)
 |    |-- auth: string (nullable = true)
 |    |-- mag: double (nullable = true)
 |    |-- magtype: string (nullable = true)
 |    |-- unid: string (nullable = true)
 |-- type: string (nullable = true)



### Tahap 4: Flattening (Membuka Nested GeoJSON)
Struktur asli USGS berupa *JSON bersarang* (misal: `geometry.coordinates[0]`). Kita harus mengeluarkannya menjadi baris dan kolom yang datar (*tabular*).

In [24]:
df_clean = df_raw.select(
    to_timestamp(col("properties.time")).alias("time"),
    col("properties.lat").cast(DoubleType()).alias("latitude"),
    col("properties.lon").cast(DoubleType()).alias("longitude"),
    col("properties.depth").cast(DoubleType()).alias("depth"),
    col("properties.mag").cast(DoubleType()).alias("mag"),
    col("properties.flynn_region").alias("place"),
    col("properties.auth").alias("auth"),
    col("properties.evtype").alias("type")
)


### Tahap 5: Penyaringan (*Filtering*) & Penghapusan Outliers
Hanya mengambil kejadian berjenis `earthquake`. Menghapus baris yang kosong (`null`), menghapus duplikat, dan membuang koordinat yang tidak rasional (misal nilai magnitudo > 10).

In [25]:
df_clean = df_clean.filter(col("type") == "ke").drop("type")
df_clean = df_clean.dropna(subset=["latitude", "longitude", "depth", "mag"])
df_clean = df_clean.dropDuplicates()

df_clean = df_clean.filter((col("depth") >= 0) & (col("depth") <= 700))
df_clean = df_clean.filter((col("mag") > 2.5) & (col("mag") <= 10))
df_clean = df_clean.filter((col("latitude") >= -90) & (col("latitude") <= 90))
df_clean = df_clean.filter((col("longitude") >= -180) & (col("longitude") <= 180))

### Tahap 6: Ekstraksi Nama Negara
Mencacah teks panjang di kolom `place` (contoh: *10 km W of Jakarta, Indonesia*) lalu mengambil kata terakhirnya menjadi fitur baru bernama `country`.

In [26]:
from pyspark.sql.functions import length, when, element_at, upper, col, trim, split, regexp_replace

def extract_country(place_col):
    return upper(trim(element_at(split(place_col, ","), -1)))

df_clean = df_clean.withColumn("raw_country", extract_country(col("place")))

# Buang awalan dan akhiran standar F-E agar nama negara bersih
prefixes = "^(EASTERN |WESTERN |NORTHERN |SOUTHERN |CENTRAL |NORTHWESTERN |SOUTHWESTERN |NORTHEASTERN |SOUTHEASTERN |OFF COAST OF |NEAR COAST OF |SOUTH OF |NORTH OF |EAST OF |WEST OF |OFFSHORE )"
suffixes = "( REGION| ISLANDS| ISLAND)$"
df_clean = df_clean.withColumn("raw_country", regexp_replace(col("raw_country"), prefixes, ""))
df_clean = df_clean.withColumn("raw_country", regexp_replace(col("raw_country"), suffixes, ""))
df_clean = df_clean.withColumn("raw_country", when(col("raw_country") == "P.N.G.", "PAPUA NEW GUINEA").otherwise(col("raw_country")))

us_states_regex = "ALASKA|CALIFORNIA|HAWAII|NEVADA|TEXAS|WASHINGTON|OREGON|IDAHO|MONTANA|WYOMING|UTAH|COLORADO|NEW MEXICO|ARIZONA|OKLAHOMA|KANSAS|NEBRASKA|SOUTH DAKOTA|NORTH DAKOTA|PUERTO RICO|VIRGIN ISLANDS|GUAM"
indo_regions_regex = "BALI|MOLUCCA|BANDA|JAVA|CELEBES|TIMOR|ARAFURA|FLORES|SAVU|HALMAHERA|SERAM|MAKASSAR|SUNDA|TALAUD|SUMATRA|MINAHASA|IRIAN JAYA|NIAS|SUMBAWA|LOMBOK|INDONESIA"
russia_regions_regex = "KAMCHATKA|KURIL|SAKHALIN|SIBERIA|BAYKAL|CAUCASUS|URAL|KOMANDORSKIYE|RUSSIA"

df_clean = df_clean.withColumn("country",
    when(col("raw_country").rlike(indo_regions_regex), "Indonesia")
    .when(col("raw_country").rlike(us_states_regex), "United States")
    .when(col("raw_country").rlike(russia_regions_regex), "Russia")
    .otherwise(col("raw_country"))
).drop("raw_country")


### Tahap 7: Transformasi Spasial 3D (Anti-Distorsi) & Skewness Log
- **Depth Log:** Meratakan kelengkungan ekstrem dari sebaran data kedalaman gempa.
- **Cartesian XYZ:** Mengonversi Koordinat (Bujur/Lintang) menjadi bentuk geometri tiga dimensi (3D). Ini mencegah kesalahan klastering K-Means yang mengira titik koordinat `-179` sangat jauh dengan `+179`.

In [27]:
df_clean = df_clean.withColumn("depth_log", log1p(col("depth")))

df_clean = df_clean.withColumn("lat_rad", radians(col("latitude"))) \
                   .withColumn("lon_rad", radians(col("longitude"))) \
                   .withColumn("x", cos(col("lat_rad")) * cos(col("lon_rad"))) \
                   .withColumn("y", cos(col("lat_rad")) * sin(col("lon_rad"))) \
                   .withColumn("z", sin(col("lat_rad"))) \
                   .drop("lat_rad", "lon_rad")

print(f"Sisa Data Bersih: {df_clean.count()} baris")

Sisa Data Bersih: 73219 baris


### Tahap 8: Feature Engineering (StandardScaler)
Menggabungkan seluruh kolom angka spasial (x, y, z) menjadi sebuah `Vector` raksasa, lalu menskalakan nilainya dengan **StandardScaler** agar skala 3D setara dan tidak terdistorsi jarak absolutnya.


In [28]:
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="raw_features")
df_featured = assembler.transform(df_clean)

scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(df_featured)
df_final = scaler_model.transform(df_featured)

df_final.select("country", "scaled_features").show(5, truncate=False)

+-----------+-----------------------------------------------------------------------------------------------------+
|country    |scaled_features                                                                                      |
+-----------+-----------------------------------------------------------------------------------------------------+
|Indonesia  |[-1.0106575607245152,1.1284854745926312,-0.46273256601746854,-0.6210237983996438,0.18585783335163733]|
|ICELAND    |[1.0132324185257509,-0.16825945002295467,1.6275757525052124,-0.8911306684218928,-0.25061885509543935]|
|GREECE     |[1.6637187222239003,0.41909125280630266,0.9654198607503358,-0.3925803272471107,-1.2690644614719502]  |
|EL SALVADOR|[0.2328614305907049,-1.3392635194198146,0.04831776458830906,0.38460140256845426,0.47684229231635444] |
|ITALY      |[1.5485100719100422,0.22214950106743295,1.212635970107519,-0.8297895515550909,0.7678267512810723]    |
+-----------+-----------------------------------------------------------

### Tahap 9: Menyimpan ke MongoDB
Hasil akhirnya disimpan di koleksi `clean_earthquakes`. Secara _default_ kita atur `mode("overwrite")` agar data lama terhapus jika Anda mengulang kode ini dari awal.

In [29]:
from pyspark.ml.functions import vector_to_array

# Konversi Vector ke Array agar bisa disimpan ke MongoDB
df_final = df_final.withColumn("raw_features", vector_to_array("raw_features"))
df_final = df_final.withColumn("scaled_features", vector_to_array("scaled_features"))

df_final.write.format("mongodb") \
    .mode("overwrite") \
    .option("database", MONGO_DB) \
    .option("collection", MONGO_CLEAN_COL) \
    .save()

print("Data Sukses Disimpan ke MongoDB Clean Collection!")

Data Sukses Disimpan ke MongoDB Clean Collection!
